# NISAR SM inversion

De Zan coherence/phase-closure soil-moisture inversion (`calc_SM_coherences`/
`invert_sm` from `INSAR4SM-main`), **SoilGrids texture, HH, both depths**
(5cm main / 20cm secondary -- same dry-anchor date for both, only the
in-situ depth read for the dry-state value changes, so they stay a
controlled comparison). SoilGrids alone anchors the inversion; in-situ ISMN
texture isn't used. HH polarization only, but `"HV"` can also be added.

**Dry-date selection is in-situ-driven, not ERA5-based like INSAR4SM's own
pipeline** (Section 5): the acquisition with the lowest in-situ SM reading
becomes the dry anchor, instead of a precipitation-quantile pick from ERA5.
A (station, track) group here can have just a handful of acquisitions --
too few for a quantile pick to be meaningful -- while the lowest in-situ
reading is well-defined at any sample size.

**Coherence/closure phase are read, not recomputed**, from
`nisar_data/alaska_GSLC_coherence_closure/{station}_NISAR_GSLC_coherence.csv`
(`02b_calc_nisar_coherence_closure.ipynb`) -- the full `N(N-1)/2` pairwise
matrix with `phase_<pol>`, so `coherence_<pol> * exp(i * phase_<pol>)`
reconstructs what `calc_SM_coherences`/`invert_sm` need directly. Re-run
`02b` first if that CSV still has the old sequential-only, magnitude-only
format.

**Per (station, track), not a single stack** -- a station can be covered by
several NISAR relative orbits, so the inversion runs independently per
group with enough acquisitions. **SoilGrids coverage can span more than one
tile** -- `sample_soilgrids` tries both `{band}.tif` and `{band}_ii.tif`.
**Stations with no in-situ data are skipped** wherever ground truth is
needed, falling back to a generic 14% dry state where coherence/closure
data still exists.

**Pairs with coherence below 0.3 are considered noise** and are ignored by the inversion. This value can be edited below. 

In [1]:
import sys
import os
from pathlib import Path

import pyproj
pyproj.datadir.set_data_dir(os.path.join(sys.prefix, "Library", "share", "proj"))


def require_globals(*names):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(
            f"Missing {missing} -- run the earlier setup cells first "
            "(the kernel may have been restarted)."
        )


BASE = Path.cwd()
COH_CLOSURE_DIR = BASE / "nisar_data" / "alaska_GSLC_coherence_closure"
INSITU_DIR = BASE / "ismn_data" / "alaska" / "station_data"
SOILGRIDS_DIR = BASE / "soilgrids_data" / "alaska"

if str(BASE / "INSAR4SM-main") not in sys.path:
    sys.path.append(str(BASE / "INSAR4SM-main"))

import numpy as np
import pandas as pd
import rasterio
from insar4sm.coh_funcs import calc_SM_coherences
from insar4sm.inversion import invert_sm


C:\Users\geoka\miniconda3\envs\nisar-era5\Lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


## 1. Station list

In [2]:
require_globals("BASE")

import json

stations_path = BASE / "ismn_data" / "alaska" / "stations.geojson"
with open(stations_path) as f:
    stations_geojson = json.load(f)
stations = {
    feat["properties"]["station_val"]: (
        feat["geometry"]["coordinates"][1],  # lat
        feat["geometry"]["coordinates"][0],  # lon
    )
    for feat in stations_geojson["features"]
}
print(f"Loaded {len(stations)} stations")


Loaded 20 stations


## 2. In-situ SM reader and a generic bracket-interpolator

Irregular real sensor timestamps (SNOTEL/USCRN, not a fixed hourly grid), so
`interp_insitu_at` brackets the nearest real readings and caps how far apart
they're allowed to be, rather than assuming a fixed grid -- same approach
`04_station_overview.ipynb` uses.

In [3]:
require_globals("INSITU_DIR")

MAX_GAP_HOURS = 6
DEPTH_MAIN = 0.05
DEPTH_SECONDARY = 0.20

def read_insitu(station, depth):
    path = INSITU_DIR / f"{station}_ts_{depth:.2f}.csv"
    if not path.exists():
        return None
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    return df.iloc[:, 0].sort_index()  # single sm_<depth> column, whatever it's named


def interp_insitu_at(series, ts):
    before = series.loc[:ts]
    after = series.loc[ts:]
    if before.empty or after.empty:
        return np.nan
    t0, v0 = before.index[-1], before.iloc[-1]
    t1, v1 = after.index[0], after.iloc[0]
    if t0 == t1:
        return float(v0)
    gap_hours = (t1 - t0).total_seconds() / 3600.0
    if gap_hours > MAX_GAP_HOURS:
        return np.nan
    frac = (ts - t0).total_seconds() / (t1 - t0).total_seconds()
    return float(v0 + frac * (v1 - v0))

## 3. Soil texture: SoilGrids

**MANUAL STEP:** at [soilgrids.org](https://soilgrids.org), use the download
tool to get **sand** and **clay** as GeoTIFF, at the **0-5cm** and **15-30cm**
depth intervals, over a bbox covering every station in your AOI. SoilGrids
names the downloaded file after the query (a WCS coverage ID), not something
this notebook recognizes -- rename each file to
`{sand|clay}_{0to5cm|15to30cm}.tif` (e.g. `sand_0to5cm.tif`,
`clay_15to30cm.tif`) and place all 4 directly under `soilgrids_data/<AOI>/`
(`SOILGRIDS_DIR` below).

If your stations don't all fit in one export's bbox, download a second tile
covering the rest and name those 4 files with an `_ii` suffix instead
(`sand_0to5cm_ii.tif`, etc.) -- `sample_soilgrids` checks both.

`sample_soilgrids` reads the 0-5cm layer for the 5cm-anchor variant and the
15-30cm layer for the 20cm-anchor variant, returning % (raw g/kg / 10).
In-situ ISMN texture is never used here as an alternative.

In [4]:
require_globals("SOILGRIDS_DIR", "stations")


def sample_soilgrids(band_stem, lon, lat):
    for suffix in ("", "_ii"):
        path = SOILGRIDS_DIR / f"{band_stem}{suffix}.tif"
        if not path.exists():
            continue
        with rasterio.open(path) as src:
            row, col = src.index(lon, lat)
            if not (0 <= row < src.height and 0 <= col < src.width):
                continue
            val = float(src.read(1)[row, col])
            if 0 < val <= 1000:  # plausible g/kg range; SoilGrids has no explicit nodata sentinel here
                return val / 10.0  # g/kg -> %
    return None


def porosity_from_texture(sand_pct, clay_pct):
    return 100 * (0.332 - 7.251e-4 * sand_pct + 0.1276 * np.log10(clay_pct))


TEXTURE_LAYERS = {DEPTH_MAIN: "0to5cm", DEPTH_SECONDARY: "15to30cm"}

texture_soilgrids = {}
for station, (lat, lon) in stations.items():
    sg_by_depth = {}
    for depth, layer in TEXTURE_LAYERS.items():
        sand_sg = sample_soilgrids(f"sand_{layer}", lon, lat)
        clay_sg = sample_soilgrids(f"clay_{layer}", lon, lat)
        if sand_sg is not None and clay_sg is not None:
            sg_by_depth[depth] = {"sand_pct": sand_sg, "clay_pct": clay_sg, "porosity_pct": porosity_from_texture(sand_sg, clay_sg)}
    if sg_by_depth:
        texture_soilgrids[station] = sg_by_depth

print(f"SoilGrids texture: {len(texture_soilgrids)}/{len(stations)} station(s)")

SoilGrids texture: 19/20 station(s)


## 4. Load the precomputed coherence/closure CSVs, build per-(station, track, pol) matrices

Reads `{station}_NISAR_GSLC_coherence.csv` directly (full pairwise matrix,
magnitude + phase per pol) -- no raw pixel access. `slc_dates_group[(station,
track)]` is that group's sorted acquisition dates; `inc_angle_group` its
mean local incidence angle. `coh_full_ds_group`/`ph_ds_group` are the
`(1, n, n)` matrices `calc_SM_coherences`/`invert_sm` expect, keyed
`(station, track, pol)` -- built per polarization in `POLARIZATIONS` (HH
only by default).

A coherence threshold is introduced, so pairs below it do not get factored in the inversion. This value is set at 0.3, but it can be changed in the cell below. 

In [5]:
require_globals("COH_CLOSURE_DIR", "stations")

POLARIZATIONS = ("HH",)
COH_THRESHOLD = 0.3  # below this, a pair is excluded from the coherence-fit term

slc_dates_group = {}
inc_angle_group = {}
coh_full_ds_group = {}  # (station, track, pol) -> (1, n, n)
ph_ds_group = {}        # (station, track, pol) -> (1, n, n)

for station in stations:
    coh_path = COH_CLOSURE_DIR / f"{station}_NISAR_GSLC_coherence.csv"
    if not coh_path.exists():
        continue
    df = pd.read_csv(coh_path)
    for pol in POLARIZATIONS:
        if f"phase_{pol}" not in df.columns:
            raise RuntimeError(
                f"{coh_path} has no 'phase_{pol}' column -- it was written before "
                "nisar_coherence_closure.py's process_station() was fixed to save "
                "full-pairwise-matrix coherence + phase (not just sequential-pair, "
                "magnitude-only). Re-run 02b_calc_nisar_coherence_closure.ipynb first to "
                "regenerate alaska_GSLC_coherence_closure/ with the current code, "
                "then re-run this notebook."
            )
    df["date_1"] = pd.to_datetime(df["date_1"])
    df["date_2"] = pd.to_datetime(df["date_2"])

    for track, df_track in df.groupby("orbit_num"):
        dates = sorted(set(df_track["date_1"]) | set(df_track["date_2"]))
        n = len(dates)
        if n < 3:
            continue  # can't form a phase-closure triplet
        date_idx = {d: i for i, d in enumerate(dates)}

        slc_dates_group[(station, track)] = dates
        inc_angle_group[(station, track)] = float(df_track["local_incidence_angle_deg"].mean())

        for pol in POLARIZATIONS:
            coh_full_ds = np.full((1, n, n), np.nan, dtype=float)
            ph_ds = np.zeros((1, n, n), dtype=np.complex128)
            for i in range(n):
                ph_ds[0, i, i] = 1.0 + 0j

            for row in df_track.itertuples():
                i, j = date_idx[row.date_1], date_idx[row.date_2]
                coh_val = getattr(row, f"coherence_{pol}")
                phase_val = getattr(row, f"phase_{pol}")
                if np.isnan(coh_val) or np.isnan(phase_val):
                    continue
                ph_val = coh_val * np.exp(1j * phase_val)
                ph_ds[0, i, j] = ph_val
                ph_ds[0, j, i] = np.conj(ph_val)
                if coh_val < COH_THRESHOLD:
                    coh_full_ds[0, i, j] = np.nan
                    coh_full_ds[0, j, i] = np.nan
                else:
                    coh_full_ds[0, i, j] = coh_val
                    coh_full_ds[0, j, i] = coh_val

            coh_full_ds_group[(station, track, pol)] = coh_full_ds
            ph_ds_group[(station, track, pol)] = ph_ds

print(f"{len(slc_dates_group)} (station, track) group(s) with >= 3 acquisitions")
for key, dates in slc_dates_group.items():
    print(f"  {key[0]} track {key[1]}: {len(dates)} acquisition(s)")


83 (station, track) group(s) with >= 3 acquisitions
  MooseInc track 15: 4 acquisition(s)
  MooseInc track 78: 5 acquisition(s)
  MooseInc track 87: 5 acquisition(s)
  MooseInc track 150: 5 acquisition(s)
  AtigunPass track 63: 5 acquisition(s)
  AtigunPass track 73: 4 acquisition(s)
  AtigunPass track 102: 4 acquisition(s)
  AtigunPass track 164: 5 acquisition(s)
  Coldfoot track 63: 5 acquisition(s)
  Coldfoot track 73: 4 acquisition(s)
  Coldfoot track 145: 5 acquisition(s)
  Coldfoot track 164: 5 acquisition(s)
  EagleSummit track 15: 4 acquisition(s)
  EagleSummit track 63: 5 acquisition(s)
  EagleSummit track 87: 5 acquisition(s)
  EagleSummit track 116: 4 acquisition(s)
  EagleSummit track 135: 5 acquisition(s)
  EagleSummit track 164: 5 acquisition(s)
  GobblersKnob track 63: 5 acquisition(s)
  GobblersKnob track 73: 4 acquisition(s)
  GobblersKnob track 145: 5 acquisition(s)
  GobblersKnob track 164: 5 acquisition(s)
  GraniteCrk track 15: 4 acquisition(s)
  GraniteCrk track 6

## 5. Per-(station, track) dry-date selection (5cm-anchored)

The acquisition with the lowest interpolated 5cm in-situ SM reading becomes
that group's dry anchor (see intro for why not ERA5). Reused for both depth
variants -- the 20cm run only re-reads the in-situ value at 20cm for the
dry-state (Section 6).

In [6]:
require_globals("slc_dates_group")

dry_idx_group = {}
dry_acq_time_group = {}
dry_bands_inds_group = {}
SM_sorting_group = {}
SM_index_init_group = {}
sm_dry_state_5cm_group = {}

for key, dates in slc_dates_group.items():
    station, track = key
    n = len(dates)
    series = read_insitu(station, DEPTH_MAIN)
    vals = None
    if series is not None:
        vals = np.array([interp_insitu_at(series, ts) for ts in dates])
    if vals is None or np.all(np.isnan(vals)):
        idx = 0
        sorting = np.arange(n)
        dry_val_pct = 14.0
    else:
        idx = int(np.nanargmin(vals))
        sorting = np.argsort(vals)  # NaN entries sort last, numpy's default
        dry_val_pct = float(vals[idx] * 100)
    dry_idx_group[key] = idx
    dry_acq_time_group[key] = dates[idx].to_pydatetime()
    dry_bands_inds_group[key] = np.array([idx], dtype=int)
    SM_sorting_group[key] = sorting
    init = np.empty(n)
    init[sorting] = np.linspace(10, 30, n)
    SM_index_init_group[key] = init
    sm_dry_state_5cm_group[key] = dry_val_pct

print("Per-(station, track) dry-anchor acquisition and 5cm dry state:")
for key in slc_dates_group:
    print(f"  {key[0]} track {key[1]}: {dry_acq_time_group[key]}  {sm_dry_state_5cm_group[key]:.2f}%")


Per-(station, track) dry-anchor acquisition and 5cm dry state:
  MooseInc track 15: 2026-07-09 05:52:56  40.80%
  MooseInc track 78: 2026-07-13 14:30:44  40.80%
  MooseInc track 87: 2026-07-14 05:44:37  40.80%
  MooseInc track 150: 2026-07-06 14:22:25  40.80%
  AtigunPass track 63: 2026-07-12 13:34:50  9.20%
  AtigunPass track 73: 2026-07-13 06:23:52  8.12%
  AtigunPass track 102: 2026-07-03 06:39:55  12.67%
  AtigunPass track 164: 2026-07-07 13:43:11  10.00%
  Coldfoot track 63: 2026-08-17 13:34:48  18.10%
  Coldfoot track 73: 2026-07-13 06:23:52  19.88%
  Coldfoot track 145: 2026-08-11 06:15:48  19.00%
  Coldfoot track 164: 2026-08-24 13:43:09  18.60%
  EagleSummit track 15: 2026-08-14 05:51:07  8.97%
  EagleSummit track 63: 2026-07-12 13:34:14  14.26%
  EagleSummit track 87: 2026-07-14 05:43:01  11.40%
  EagleSummit track 116: 2026-07-04 05:59:29  14.80%
  EagleSummit track 135: 2026-08-10 13:25:53  13.24%
  EagleSummit track 164: 2026-08-12 13:42:32  11.50%
  GobblersKnob track 63:

## 6. 20cm dry state (same anchor date, different in-situ depth read)

In [7]:
require_globals("dry_acq_time_group")

sm_dry_state_20cm_group = {}
for key in slc_dates_group:
    station, track = key
    series = read_insitu(station, DEPTH_SECONDARY)
    if series is not None:
        val = interp_insitu_at(series, dry_acq_time_group[key])
        if np.isnan(val):
            val = series.min()
        sm_dry_state_20cm_group[key] = float(val * 100)
    else:
        sm_dry_state_20cm_group[key] = 14.0


## 7. Run the inversion

`run_inversion` is the shared body for both variants -- one
`calc_SM_coherences`/`invert_sm` call, parameterized by which texture
(`clay_pct`/`sand_pct`/`porosity_pct`) and which depth's `sm_dry_state` to use.

In [8]:
require_globals(
    "coh_full_ds_group", "ph_ds_group", "slc_dates_group", "inc_angle_group",
    "SM_sorting_group", "SM_index_init_group", "dry_bands_inds_group", "dry_acq_time_group",
)


def run_inversion(key, pol, sm_dry_state, clay_pct, sand_pct, porosity_pct):
    n = len(slc_dates_group[key])
    slc_dates_arr = np.array([d.to_pydatetime() for d in slc_dates_group[key]], dtype=object)

    coh_sm, _, _ = calc_SM_coherences(
        Datetimes=slc_dates_arr,
        SM_sorting=SM_sorting_group[key],
        n_bands=n,
        n_ds=1,
        ind_DS=0,
        coh_full_DS=coh_full_ds_group[(key[0], key[1], pol)],
        band_start=0,
        band_end=n,
        n_dry_bands=1,
        max_coh_flag=True,
        simple_exp_model=False,
        sm_only_mode=True,
    )

    sm_pix = invert_sm(
        ph_DS=ph_ds_group[(key[0], key[1], pol)],
        coh_sm=coh_sm[0],
        SM_index=SM_index_init_group[key],
        dry_bands_inds=dry_bands_inds_group[key],
        slc_dates=slc_dates_arr,
        DS_ind=0,
        driest_date=dry_acq_time_group[key],
        inc_DS=np.array([inc_angle_group[key]]),
        band_start=0,
        band_end=n,
        nbands=n,
        opt_method="SLSQP",
        opt_parms={"ftol": 1e-3, "eps": 0.01, "maxiter": 50},
        ph_closure_dist=5,
        weight_factor=1.0,
        sm_dry_state=sm_dry_state,
        freq_GHz=1.2575,
        clay_pct=clay_pct,
        sand_pct=sand_pct,
    )
    return np.clip(sm_pix, 0, porosity_pct)


sm_map = {}  # (station, track, texture_source, depth_label, pol) -> sm array
for key in slc_dates_group:
    station, track = key

    for pol in POLARIZATIONS:
        if station in texture_soilgrids:
            for depth, dry_state_group, depth_label in (
                (DEPTH_MAIN, sm_dry_state_5cm_group, "5cm"),
                (DEPTH_SECONDARY, sm_dry_state_20cm_group, "20cm"),
            ):
                tex = texture_soilgrids[station].get(depth)
                if tex is None:
                    continue
                sm_pix = run_inversion(key, pol, dry_state_group[key], tex["clay_pct"], tex["sand_pct"], tex["porosity_pct"])
                sm_map[(station, track, "soilgrids", depth_label, pol)] = sm_pix
                print(f"  {station} track {track} [soilgrids, {depth_label}, {pol}]: SM = {sm_pix.round(2)} %")

print(f"\nInversion complete: {len(sm_map)} (station, track, texture, depth, pol) result(s)")

  AtigunPass track 63 [soilgrids, 5cm, HH]: SM = [34.82 29.97  9.2  42.71 12.15] %
  AtigunPass track 63 [soilgrids, 20cm, HH]: SM = [35.51 30.86 10.46 41.87 13.52] %


  AtigunPass track 73 [soilgrids, 5cm, HH]: SM = [14.45 26.51  8.12 24.91] %
  AtigunPass track 73 [soilgrids, 20cm, HH]: SM = [18.12 31.95 10.5  30.3 ] %


  AtigunPass track 102 [soilgrids, 5cm, HH]: SM = [12.67 10.13 26.63 24.77] %
  AtigunPass track 102 [soilgrids, 20cm, HH]: SM = [19.2  15.32 27.43 24.56] %


  AtigunPass track 164 [soilgrids, 5cm, HH]: SM = [17.84 10.   14.21 22.65 24.69] %
  AtigunPass track 164 [soilgrids, 20cm, HH]: SM = [22.93 14.04 18.85 27.97 30.21] %


  Coldfoot track 63 [soilgrids, 5cm, HH]: SM = [21.24 10.96  7.68 17.28 18.1 ] %
  Coldfoot track 63 [soilgrids, 20cm, HH]: SM = [27.41 16.35 12.39 23.45 23.72] %


...

  Kenai-29-ENE track 15 [soilgrids, 5cm, HH]: SM = [13.09 26.47 19.56  4.86] %
  Kenai-29-ENE track 15 [soilgrids, 20cm, HH]: SM = [26.2  11.81 33.86 15.45] %


  Kenai-29-ENE track 49 [soilgrids, 5cm, HH]: SM = [14.1  42.55 29.43 42.55] %
  Kenai-29-ENE track 49 [soilgrids, 20cm, HH]: SM = [24.58  9.94  0.    9.55] %


  Kenai-29-ENE track 87 [soilgrids, 5cm, HH]: SM = [11.94 12.37 22.01 25.16 18.84] %
  Kenai-29-ENE track 87 [soilgrids, 20cm, HH]: SM = [14.58 25.1  24.42 29.49 21.12] %


  Kenai-29-ENE track 150 [soilgrids, 5cm, HH]: SM = [10.03 13.1  42.55 27.16 18.17] %


  Kenai-29-ENE track 150 [soilgrids, 20cm, HH]: SM = [ 5.58 24.7  38.25 17.44 38.76] %

Inversion complete: 158 (station, track, texture, depth, pol) result(s)


## 8. Export SM estimates

Writes `sm_map` to a CSV (station, track, texture_source, depth, pol, date,
sm_pct, is_dry_anchor) so other notebooks (e.g. the station-overview one)
can use these estimates without re-running the inversion. `is_dry_anchor`
flags the acquisition forced to match the in-situ dry state by the
inversion's own equality constraint (always excluded from any downstream RMSE/bias/R comparison, e.g. in `04`).

In [9]:
require_globals("sm_map", "slc_dates_group", "dry_idx_group")

export_rows = []
for (station, track, texture_source, depth_label, pol), sm_pix in sm_map.items():
    dates = slc_dates_group[(station, track)]
    dry_idx = dry_idx_group[(station, track)]
    for i, (date, sm_pct) in enumerate(zip(dates, sm_pix)):
        export_rows.append({
            "station": station, "track": track, "texture_source": texture_source,
            "depth": depth_label, "pol": pol, "date": date, "sm_pct": sm_pct,
            "is_dry_anchor": i == dry_idx,
        })

sm_estimates_df = pd.DataFrame(export_rows).sort_values(["station", "track", "depth", "date"])
sm_estimates_path = BASE / "nisar_data" / "alaska_NISAR_SM_estimates.csv"
sm_estimates_df.to_csv(sm_estimates_path, index=False)
print(f"Wrote {len(sm_estimates_df)} row(s) to {sm_estimates_path}")

Wrote 732 row(s) to D:\Docs\RS_2023\NISAR4SM\nisar_data\alaska_NISAR_SM_estimates.csv


<hr>

## Summary

`sm_map` holds the per-(station, track, texture_source, depth, pol) SM
estimate array. Exported below (Section 8) for the station-overview
notebook (`04`), which does the in-situ/SME2 comparison, screening, and
visualization. Re-run after `alaska_GSLC_coherence_closure/`, SoilGrids, or
SME2 data changes -- everything here is purely local, no network cost.

<hr>